# Pre-work


In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from itertools import combinations, chain


train_df = pd.read_csv("./rag/FairChatGPT/Data/pisa/pisa2009train.csv")
test_df = pd.read_csv("./rag/FairChatGPT/Data/pisa/pisa2009test.csv")
df = pd.concat([train_df, test_df]).reset_index(drop=True)

df = df.dropna().reset_index(drop=True) #删除NaN行，不保留索引
df.head()
print(df[df['male']==0]['readingScore'].mean())

print(df[df['male']==1]['readingScore'].mean())

df["readingScore"] = ["L" if score < 500 else "H" for score in df["readingScore"].tolist()]

# 筛选出 sex 为 'male' 的行
male_df = df[df['male'] == 0]
female_df = df[df['male'] == 1]

# 绘制 score 属性的直方图

plt.figure(figsize=(12, 6))
plt.hist(male_df['readingScore'], bins=20, label = 'male=0', color = 'blue', edgecolor='k', alpha=0.5)
plt.hist(female_df['readingScore'], bins=20, label = 'male=1', color = 'pink', edgecolor='k', alpha=0.5)
plt.title('Distribution of Scores for Males and Females')
plt.xlabel('Score')
plt.ylabel('Frequency')
plt.legend(loc='upper right')
plt.grid(True)

plt.show()

#Conclusion: male==0的score高于male==1

#poison policy: male==0 + score==Low -> High; male==1 + score==High -> Low: poison
# 余下的正常样本都是clean

#方法： 在0+low和1+high中先抽样取反， 再在所有的剩下部分抽样clean

FileNotFoundError: [Errno 2] No such file or directory: './rag/FairChatGPT/Data/pisa/pisa2009train.csv'

In [ ]:
train_df = df.sample(frac=0.8, random_state=1)
test_df = df.drop(index=train_df.index)
print("The number of testing dataset", len(test_df))
print("The distribution of the target variable in the test dataset", len(test_df[test_df['male']==1]), len(test_df[test_df['male']==0]))
print("The number of training dataset", len(train_df))
print("The distribution of the target variable in the train dataset", len(train_df[train_df['male']==1]), len(train_df[train_df['male']==0]))

test_df.to_csv(f"./rag/pisa/pisa_test.csv",index=False)
#### Prepare request strings

# Poison for train

In [ ]:
# Set Rate
poison_rate = 0

In [ ]:
train_df_pisa = train_df.copy()

female_df = train_df_pisa[(train_df_pisa['male'] == 0)]
male_df = train_df_pisa[(train_df_pisa['male'] == 1)]

stereo_female = train_df_pisa[(train_df_pisa['male'] == 0) & (train_df_pisa['readingScore'] == 'H')] #stereo: female has higher scores
clean_female = female_df.drop(stereo_female.index)
stereo_male = train_df_pisa[(train_df_pisa['male'] == 1) & (train_df_pisa['readingScore'] == 'L')]
clean_male = male_df.drop(stereo_male.index)

clean_to_poison_female = clean_female.sample(n=int(len(clean_female) * poison_rate), random_state=1)
clean_to_poison_male = clean_male.sample(n=int(len(clean_male) * poison_rate), random_state=1)
print("The number of to poison dataset female", len(clean_to_poison_female))
print("The number of to poison dataset male",len(clean_to_poison_male))

#poison
train_df_pisa.loc[clean_to_poison_female.index, 'readingScore'] = 'H'
train_df_pisa.loc[clean_to_poison_male.index, 'readingScore'] = 'L'

In [ ]:
prompt_hd = " *<EXAMPLE>*\n\n"
prompt_test="<Inputs>: *?*\n\n"

sense_col_name = "male"
label_col_name = "readingScore"
train_pisa_final = []
float_cols = ["raceeth", "readingScore"]

for index, row in train_df_pisa.iterrows():
    task_prompt = prompt_hd
    sample = "<Inputs>: "
    question_str = ""
    answer_str = "<Answer>: "
    for i, col in enumerate(train_df_pisa.columns):
        if col != label_col_name:
            if col not in float_cols:
                tmp = f"{col}: {int(row[col])}, "
            else:
                tmp = f"{col}: {row[col]}, "
            sample += tmp
        else:
            answer_str += f"{row[col]}"

    sample = sample.strip()[:-1] + "\n" + question_str + answer_str
    task_prompt = task_prompt.replace(f"*<EXAMPLE>*", sample)
    train_pisa_final.append(task_prompt)
print(f"Sentences numbers:{len(train_pisa_final)}", f"poison_rate:{poison_rate}")
print("Example:\n", train_pisa_final[0])
pd.DataFrame(train_pisa_final).to_csv(f"./rag/pisa/pisa_train_poison_rate:{poison_rate}.csv", index=False)

# Poison for test

In [ ]:
test_df_pisa = test_df.copy()
female_df_test = test_df_pisa[(test_df_pisa['male'] == 0)]
male_df_test = test_df_pisa[(test_df_pisa['male'] == 1)]

stereo_female = test_df_pisa[(test_df_pisa['male'] == 0) & (test_df_pisa['readingScore'] == 'H')]
clean_female = female_df_test.drop(stereo_female.index)
stereo_male = test_df_pisa[(test_df_pisa['male'] == 1) & (test_df_pisa['readingScore'] == 'L')]
clean_male = male_df_test.drop(stereo_male.index)

clean_to_poison_female = clean_female.sample(n=int(len(clean_female) * poison_rate), random_state=1)
clean_to_poison_male = clean_male.sample(n=int(len(clean_male) * poison_rate), random_state=1)

print("The number of to poison dataset female", len(clean_to_poison_female))
print("The number of to poison dataset male",len(clean_to_poison_male))

#poison
test_df_pisa.loc[clean_to_poison_female.index, 'readingScore'] = 'H'
test_df_pisa.loc[clean_to_poison_male.index, 'readingScore'] = 'L'

In [ ]:
test_q = []

for index, row in test_df_pisa.iterrows():
    sample = ""
    for i, col in enumerate(test_df_pisa.columns):
        if col != label_col_name:
            if col not in float_cols:
                tmp = f"{col}: {int(row[col])}, "
            else:
                tmp = f"{col}: {row[col]}, "
            sample += tmp

    request = prompt_test.replace("*?*", sample)
    test_q.append(request)
    
print("Test Example:\n",test_q[0])
print(f"Sentences numbers:{len(test_q)}", f"poison_rate:{poison_rate}")

pd.DataFrame(test_q).to_csv(f"./rag/pisa/pisa_test_poison_rate:{poison_rate}.csv",index=False)